# Global Food Security Pipeline

## Phase 0 Spike

Goal: validate all data sources *before* committing to an architecture.

Sources to validate:
1. FAOSTAT bulk download: Food Security Indicators (FS dataset)
2. FAOSTAT bulk download: Food Price Index (FP dataset)
3. World Bank API: GDP per capita + poverty headcount
4. Country code alignment across sources
5. Open-Meteo: climate data feasibility

## 0. Install dependencies

In [1]:
import io
import zipfile
import requests
import pandas as pd
import wbgapi as wb

# A small set of countries for quick validation
# Deliberately diverse: rich, poor, large, small, different regions
SAMPLE_COUNTRIES = ['India', 'Brazil', 'Nigeria', 'Germany', 'Ethiopia', 'United States of America']

---
## 1. FAOSTAT: Food Security Indicators (dataset code: FS)

This contains: undernourishment prevalence, food insecurity rates, dietary energy supply.
Using bulk download (zip CSV), no auth required.

In [2]:
FS_URL = "https://bulks-faostat.fao.org/production/Food_Security_Data_E_All_Data_(Normalized).zip"

print(f"Downloading food security data from FAO...")
response = requests.get(FS_URL, timeout=120)
print(f"Status: {response.status_code} | Size: {len(response.content) / 1e6:.1f} MB")

Status: 200 | Size: 2.2 MB


In [3]:
# Unzip and read the main CSV
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    print("Files inside the zip:", z.namelist())

    # Check the main data file
    main_file = [f for f in z.namelist() if f.endswith('Normalized).csv')][0]
    print(f"Reading: {main_file}")
    with z.open(main_file) as f:
        fao_fs = pd.read_csv(f, encoding='latin-1')

print(f"\nShape: {fao_fs.shape}")
print(f"Columns: {list(fao_fs.columns)}")

Files inside the zip: ['Food_Security_Data_E_All_Data_(Normalized).csv', 'Food_Security_Data_E_AreaCodes.csv', 'Food_Security_Data_E_Elements.csv', 'Food_Security_Data_E_Flags.csv', 'Food_Security_Data_E_ItemCodes.csv']
Reading: Food_Security_Data_E_All_Data_(Normalized).csv

Shape: (279470, 13)
Columns: ['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item', 'Element Code', 'Element', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Note']


In [4]:
# Understand the structure
fao_fs.head(5)

,Area Code,Area Code (M49),Area,Item Code,Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
0,2,'004,Afghanistan,21010,Average dietary energy supply adequacy (percen...,6121,Value,20002002,2000-2002,%,87,E,NaN
1,2,'004,Afghanistan,21010,Average dietary energy supply adequacy (percen...,6121,Value,20012003,2001-2003,%,88,E,NaN
2,2,'004,Afghanistan,21010,Average dietary energy supply adequacy (percen...,6121,Value,20022004,2002-2004,%,91,E,NaN
3,2,'004,Afghanistan,21010,Average dietary energy supply adequacy (percen...,6121,Value,20032005,2003-2005,%,93,E,NaN
4,2,'004,Afghanistan,21010,Average dietary energy supply adequacy (percen...,6121,Value,20042006,2004-2006,%,95,E,NaN


In [5]:
# What indicators are available?
print("All items/indicators:")
print(fao_fs['Item'].unique())

All items/indicators:
<StringArray>
[                                                         'Average dietary energy supply adequacy (percent) (3-year average)',
                          'Dietary energy supply used in the estimation of the prevalence of undernourishment (kcal/cap/day)',
         'Dietary energy supply used in the estimation of the prevalence of undernourishment (kcal/cap/day) (3-year average)',
                           'Share of dietary energy supply derived from cereals, roots and tubers (percent) (3-year average)',
                                                                        'Average protein supply (g/cap/day) (3-year average)',
                                                    'Average supply of protein of animal origin (g/cap/day) (3-year average)',
                                                    'Gross domestic product per capita, PPP, (constant 2021 international $)',
                                                                  'Prevalen

In [6]:
# Focus on undernourishment — the key metric
undernourishment_label = 'Prevalence of undernourishment (percent) (3-year average)'

df_hunger = fao_fs[fao_fs['Item'] == undernourishment_label].copy()
print(f"Undernourishment rows: {len(df_hunger)}")
print(f"Year range: {df_hunger['Year'].min()} to {df_hunger['Year'].max()}")
print(f"Countries: {df_hunger['Area'].nunique()}")
df_hunger.head(5)

Undernourishment rows: 5727
Year range: 2000-2002 to 2022-2024
Countries: 249


,Area Code,Area Code (M49),Area,Item Code,Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
156,2,'004,Afghanistan,210041,Prevalence of undernourishment (percent) (3-ye...,6121,Value,20002002,2000-2002,%,45.8,E,NaN
157,2,'004,Afghanistan,210041,Prevalence of undernourishment (percent) (3-ye...,6121,Value,20012003,2001-2003,%,43.5,E,NaN
158,2,'004,Afghanistan,210041,Prevalence of undernourishment (percent) (3-ye...,6121,Value,20022004,2002-2004,%,38.7,E,NaN
159,2,'004,Afghanistan,210041,Prevalence of undernourishment (percent) (3-ye...,6121,Value,20032005,2003-2005,%,34.6,E,NaN
160,2,'004,Afghanistan,210041,Prevalence of undernourishment (percent) (3-ye...,6121,Value,20042006,2004-2006,%,30.6,E,NaN


In [ ]:
# Check sample countries — do they all exist?
for country in SAMPLE_COUNTRIES:
    match = df_hunger[df_hunger['Area'].str.contains(country, case=False)]
    if len(match) > 0:
        print(f"(A) {country}: {len(match)} rows, years {match['Year'].min()}-{match['Year'].max()}")
    else:
        print(f"(NA) {country}: NOT FOUND — check exact name in FAO")

(A) India: 46 rows, years 2000-2002-2022-2024
(A) Brazil: 23 rows, years 2000-2002-2022-2024
(A) Nigeria: 23 rows, years 2000-2002-2022-2024
(A) Germany: 23 rows, years 2000-2002-2022-2024
(A) Ethiopia: 23 rows, years 2000-2002-2022-2024
(A) United States of America: 23 rows, years 2000-2002-2022-2024


In [8]:
# Check values and meaning of flags
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    flag_file = [f for f in z.namelist() if f.endswith('E_Flags.csv')][0]
    print(f"\nReading flags from: {flag_file}")
    with z.open(flag_file) as f:
        flag_df = pd.read_csv(f, encoding='latin-1')
flag_df


Reading flags from: Food_Security_Data_E_Flags.csv


,Flag,Description
0,A,Official figure
1,E,Estimated value
2,O,Missing value
3,Q,Missing value; suppressed
4,X,Figure from external organization


In [9]:
# Check nulls and flag values
print("Null values per column:")
print(df_hunger.isnull().sum())

print("\nFlag distribution:")
print(df_hunger['Flag'].value_counts(dropna=False))

Null values per column:
Area Code             0
Area Code (M49)       0
Area                  0
Item Code             0
Item                  0
Element Code          0
Element               0
Year Code             0
Year                  0
Unit                  0
Value               838
Flag                  0
Note               5727
dtype: int64

Flag distribution:
Flag
E    4889
O     838
Name: count, dtype: int64


---
## 2. FAOSTAT: Food Price Index (dataset code: FPI)

The FAO Food Price Index tracks monthly changes in international food commodity prices.
Also available as bulk download.

In [10]:
# FAO Consumer Price Indices
FPI_URL = "https://bulks-faostat.fao.org/production/ConsumerPriceIndices_E_All_Data_(Normalized).zip"

print("Downloading food price index data...")
r2 = requests.get(FPI_URL, timeout=120)
print(f"Status: {r2.status_code} | Size: {len(r2.content) / 1e6:.1f} MB")

Status: 200 | Size: 2.3 MB


In [11]:
with zipfile.ZipFile(io.BytesIO(r2.content)) as z:
    print("Files:", z.namelist())
    main_file = [f for f in z.namelist() if f.endswith('Normalized).csv')][0]
    with z.open(main_file) as f:
        fao_fpi = pd.read_csv(f, encoding='latin-1')

print(f"Shape: {fao_fpi.shape}")
print(f"Columns: {list(fao_fpi.columns)}")
fao_fpi.head(5)

Files: ['ConsumerPriceIndices_E_All_Data_(Normalized).csv', 'ConsumerPriceIndices_E_AreaCodes.csv', 'ConsumerPriceIndices_E_Elements.csv', 'ConsumerPriceIndices_E_Flags.csv', 'ConsumerPriceIndices_E_ItemCodes.csv']
Shape: (237814, 15)
Columns: ['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item', 'Element Code', 'Element', 'Months Code', 'Months', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Note']


,Area Code,Area Code (M49),Area,Item Code,Item,Element Code,Element,Months Code,Months,Year Code,Year,Unit,Value,Flag,Note
0,2,'004,Afghanistan,23013,"Consumer Prices, Food Indices (2015 = 100)",6125,Value,7001,January,2000,2000,NaN,24.356332,I,base year is 2015
1,2,'004,Afghanistan,23013,"Consumer Prices, Food Indices (2015 = 100)",6125,Value,7001,January,2001,2001,NaN,29.944592,I,base year is 2015
2,2,'004,Afghanistan,23013,"Consumer Prices, Food Indices (2015 = 100)",6125,Value,7001,January,2002,2002,NaN,33.421952,I,base year is 2015
3,2,'004,Afghanistan,23013,"Consumer Prices, Food Indices (2015 = 100)",6125,Value,7001,January,2003,2003,NaN,39.967661,I,base year is 2015
4,2,'004,Afghanistan,23013,"Consumer Prices, Food Indices (2015 = 100)",6125,Value,7001,January,2004,2004,NaN,43.401939,I,base year is 2015


In [12]:
# What items/elements are available?
print("Items:", fao_fpi['Item'].unique() if 'Item' in fao_fpi.columns else 'No Item column')
print("Elements:", fao_fpi['Element'].unique() if 'Element' in fao_fpi.columns else 'No Element column')

Items: <StringArray>
[                     'Consumer Prices, Food Indices (2015 = 100)',
                   'Consumer Prices, General Indices (2015 = 100)',
                                            'Food price inflation',
              'Consumer Prices, Food Indices (2015 = 100), median',
    'Consumer Prices, Food Indices (2015 = 100), weighted average',
           'Consumer Prices, General Indices (2015 = 100), median',
 'Consumer Prices, General Indices (2015 = 100), weighted average',
                                    'Food price inflation, median',
                          'Food price inflation, weighted average']
Length: 9, dtype: str
Elements: <StringArray>
['Value']
Length: 1, dtype: str


In [13]:
# Check data for sample countries
for country in SAMPLE_COUNTRIES:
    match = fao_fpi[fao_fpi['Area'].str.contains(country, case=False)]
    if len(match) > 0:
        print(f"(A) {country}: {len(match)} rows")
    else:
        print(f"(NA) {country}: NOT FOUND")

(A) India: 915 rows
(A) Brazil: 915 rows
(A) Nigeria: 915 rows
(A) Germany: 915 rows
(A) Ethiopia: 915 rows
(A) United States of America: 915 rows


In [14]:
# Check values and meaning of flags
with zipfile.ZipFile(io.BytesIO(r2.content)) as z:
    flag_file = [f for f in z.namelist() if f.endswith('E_Flags.csv')][0]
    print(f"\nReading flags from: {flag_file}")
    with z.open(flag_file) as f:
        flag_df = pd.read_csv(f, encoding='latin-1')
flag_df


Reading flags from: ConsumerPriceIndices_E_Flags.csv


,Flag,Description
0,A,Official figure
1,E,Estimated value
2,I,Value imputed by a receiving agency
3,X,Figure from external organization


---
## 3. World Bank API: GDP per capita + Poverty headcount

Using `wbgapi`. No auth needed.
Key indicators:
- `NY.GDP.PCAP.CD` — GDP per capita (current USD)
- `SI.POV.DDAY` — Poverty headcount ratio at $2.15/day (% of population)
- `SH.XPD.CHEX.GD.ZS` — Current health expenditure (% of GDP) [useful for health mart later]

In [15]:
#help(wb.series.info)
#dir(wb.series)

In [16]:
# Fetch GDP per capita for all countries, 2000-2023
print("Fetching GDP per capita from World Bank...")
gdp_df = wb.data.DataFrame(
    'NY.GDP.PCAP.CD',
    time=range(2000, 2024),
    labels=True  # adds country names
).reset_index()

print(f"Shape: {gdp_df.shape}")
gdp_df.head(5)

Fetching GDP per capita from World Bank...
Shape: (266, 26)


,economy,Country,YR2000,YR2001,YR2002,YR2003,YR2004,YR2005,YR2006,YR2007,...,YR2014,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023
0,ZWE,Zimbabwe,562.833757,566.388746,524.936479,468.471933,469.719299,461.271388,431.034994,413.521245,...,1372.915262,1387.126326,1408.139453,3445.449410,2270.895319,2184.329239,2059.674454,2613.605421,2536.400502,2195.224921
1,ZMB,Zambia,359.429501,396.548953,393.864626,446.290105,548.685975,710.982068,1051.717149,1118.731626,...,1707.485731,1295.877887,1239.085279,1483.465773,1463.899979,1258.986198,951.644317,1127.160779,1447.123101,1330.727806
2,YEM,"Yemen, Rep.",493.235183,487.141258,513.241487,548.905883,627.632023,734.159908,809.713128,890.305768,...,1430.164210,1362.173812,975.359417,811.165970,633.887202,NaN,NaN,NaN,NaN,NaN
3,PSE,West Bank and Gaza,1476.171850,1335.553195,1156.217473,1257.698570,1422.190886,1543.701414,1570.104400,1664.245717,...,3352.112595,3272.154324,3527.613824,3620.360487,3562.330943,3656.858271,3233.568638,3678.635657,3799.955270,3455.028529
4,VIR,Virgin Islands (U.S.),NaN,NaN,30062.022505,31731.256624,35006.361440,40828.746093,41377.146601,44158.505404,...,33045.364380,34007.352941,35324.974887,35365.069304,36663.208755,38633.529892,39787.374165,42571.077737,44320.909186,NaN


In [17]:
# wbgapi returns wide format by default (columns = years)
# We need long format for the pipeline
id_cols = [c for c in gdp_df.columns if not str(c).startswith('YR')]
year_cols = [c for c in gdp_df.columns if str(c).startswith('YR')]

gdp_long = gdp_df.melt(id_vars=id_cols, value_vars=year_cols, var_name='year', value_name='gdp_per_capita')
gdp_long['year'] = gdp_long['year'].str.replace('YR', '').astype(int)

print(f"Long format shape: {gdp_long.shape}")
print(f"Null GDP values: {gdp_long['gdp_per_capita'].isnull().sum()} ({gdp_long['gdp_per_capita'].isnull().mean():.1%})")
gdp_long.head(5)

Long format shape: (6384, 4)
Null GDP values: 191 (3.0%)


,economy,Country,year,gdp_per_capita
0,ZWE,Zimbabwe,2000,562.833757
1,ZMB,Zambia,2000,359.429501
2,YEM,"Yemen, Rep.",2000,493.235183
3,PSE,West Bank and Gaza,2000,1476.171850
4,VIR,Virgin Islands (U.S.),2000,NaN


In [18]:
# Fetch poverty headcount
print("Fetching poverty headcount from World Bank...")
pov_df = wb.data.DataFrame(
    'SI.POV.DDAY',
    time=range(2000, 2024),
    labels=True
).reset_index()

pov_long = pov_df.melt(
    id_vars=[c for c in pov_df.columns if not str(c).startswith('YR')],
    value_vars=[c for c in pov_df.columns if str(c).startswith('YR')],
    var_name='year', value_name='poverty_headcount_pct'
)
pov_long['year'] = pov_long['year'].str.replace('YR', '').astype(int)

print(f"Shape: {pov_long.shape}")
print(f"Columns: {pov_long.columns}")
print(f"Null values: {pov_long['poverty_headcount_pct'].isnull().sum()} ({pov_long['poverty_headcount_pct'].isnull().mean():.1%})")

# NOTE: poverty data is sparce. It is survey-based and not available for every year and country.

Fetching poverty headcount from World Bank...
Shape: (6384, 4)
Columns: Index(['economy', 'Country', 'year', 'poverty_headcount_pct'], dtype='str')
Null values: 4212 (66.0%)


In [19]:
# What country identifier does World Bank use?
print("Sample codes:")
print(gdp_long[['economy', 'Country']].drop_duplicates().head(10) if 'Country' in gdp_long.columns else gdp_long.head(3))

Sample codes:
  economy                Country
0     ZWE               Zimbabwe
1     ZMB                 Zambia
2     YEM            Yemen, Rep.
3     PSE     West Bank and Gaza
4     VIR  Virgin Islands (U.S.)
5     VNM               Viet Nam
6     VEN          Venezuela, RB
7     VUT                Vanuatu
8     UZB             Uzbekistan
9     URY                Uruguay


---
## 4. Country Code Alignment

FAO uses its own numeric codes (and M49 codes).
World Bank uses ISO alpha-3.

We need to understand the gap before designing the seed table.

In [20]:
# Get the full list of FAO country names
fao_countries = set(fao_fs['Area'].unique())
# Filter out aggregates (FAO includes regions)

# Get World Bank country names
wb_countries = wb.economy.DataFrame().reset_index()
print(f"FAO areas: {len(fao_countries)}")
print(f"WB economies: {len(wb_countries)}")
wb_countries.head(5)

FAO areas: 249
WB economies: 266


,id,name,aggregate,longitude,latitude,region,adminregion,lendingType,incomeLevel,capitalCity
0,ABW,Aruba,False,-70.0167,12.51670,LCN,,LNX,HIC,Oranjestad
1,AFE,Africa Eastern and Southern,True,NaN,NaN,,,,,
2,AFG,Afghanistan,False,69.1761,34.52280,MEA,MNA,IDX,LIC,Kabul
3,AFW,Africa Western and Central,True,NaN,NaN,,,,,
4,AGO,Angola,False,13.2420,-8.81155,SSF,SSA,IBD,LMC,Luanda


In [21]:
wb_countries = wb.economy.DataFrame().reset_index()
wb_countries[wb_countries['aggregate'] == False]

,id,name,aggregate,longitude,latitude,region,adminregion,lendingType,incomeLevel,capitalCity
0,ABW,Aruba,False,-70.0167,12.51670,LCN,,LNX,HIC,Oranjestad
2,AFG,Afghanistan,False,69.1761,34.52280,MEA,MNA,IDX,LIC,Kabul
4,AGO,Angola,False,13.2420,-8.81155,SSF,SSA,IBD,LMC,Luanda
5,ALB,Albania,False,19.8172,41.33170,ECS,ECA,IBD,UMC,Tirane
6,AND,Andorra,False,1.5218,42.50750,ECS,,LNX,HIC,Andorra la Vella
...,...,...,...,...,...,...,...,...,...,...
261,XKX,Kosovo,False,20.9260,42.56500,ECS,ECA,IDX,UMC,Pristina
262,YEM,"Yemen, Rep.",False,44.2075,15.35200,MEA,MNA,IDX,LIC,Sana'a
263,ZAF,South Africa,False,28.1871,-25.74600,SSF,SSA,IBD,UMC,Pretoria
264,ZMB,Zambia,False,28.2937,-15.39820,SSF,SSA,IDX,LMC,Lusaka


In [22]:
wb_countries.columns

Index(['id', 'name', 'aggregate', 'longitude', 'latitude', 'region',
       'adminregion', 'lendingType', 'incomeLevel', 'capitalCity'],
      dtype='str')

In [23]:
# How many FAO country names exactly match WB country names?
wb_names = set(wb_countries['name'].values)

exact_matches = fao_countries & wb_names
fao_only = fao_countries - wb_names

print(f"Exact name matches: {len(exact_matches)}")
print(f"FAO names with NO exact WB match: {len(fao_only)}")
print("\nSample mismatches:")
# Show a sample of the mismatches
mismatch_sample = sorted(list(fao_only))[:30]
for name in mismatch_sample:
    print(f"  FAO: '{name}'")

Exact name matches: 171
FAO names with NO exact WB match: 78

Sample mismatches:
  FAO: 'Africa'
  FAO: 'Asia'
  FAO: 'Australia and New Zealand'
  FAO: 'Bahamas'
  FAO: 'Bolivia (Plurinational State of)'
  FAO: 'Caribbean'
  FAO: 'Central America'
  FAO: 'Central Asia'
  FAO: 'Central Asia and Southern Asia'
  FAO: 'China, Hong Kong SAR'
  FAO: 'China, Macao SAR'
  FAO: 'China, Taiwan Province of'
  FAO: 'China, mainland'
  FAO: 'Congo'
  FAO: 'Cook Islands'
  FAO: 'CÃ´te d'Ivoire'
  FAO: 'Democratic People's Republic of Korea'
  FAO: 'Democratic Republic of the Congo'
  FAO: 'Eastern Africa'
  FAO: 'Eastern Asia'
  FAO: 'Eastern Asia and South-eastern Asia'
  FAO: 'Eastern Europe'
  FAO: 'Egypt'
  FAO: 'Europe'
  FAO: 'Gambia'
  FAO: 'High-income economies'
  FAO: 'Iran (Islamic Republic of)'
  FAO: 'Kyrgyzstan'
  FAO: 'Land Locked Developing Countries (LLDCs)'
  FAO: 'Lao People's Democratic Republic'


In [24]:
code_col_df = fao_fs[['Area Code', 'Area Code (M49)', 'Area']].drop_duplicates().reset_index(drop=True)
code_col_df.head(10)

,Area Code,Area Code (M49),Area
0,2,'004,Afghanistan
1,3,'008,Albania
2,4,'012,Algeria
3,5,'016,American Samoa
4,6,'020,Andorra
5,7,'024,Angola
6,8,'028,Antigua and Barbuda
7,9,'032,Argentina
8,1,'051,Armenia
9,10,'036,Australia


In [25]:
# Check if FAO has ISO3 codes we can use directly
code_col = fao_fs[['Area Code (M49)']]
print(f"FAO code column: '{code_col.columns[0]}'")
print("Sample FAO country codes:")

print(fao_fs[['Area', 'Area Code (M49)']].drop_duplicates().head(15))
print("\nNOTE: M49 codes are numeric UN codes, NOT ISO alpha-3.")


FAO code column: 'Area Code (M49)'
Sample FAO country codes:
                      Area Area Code (M49)
0              Afghanistan            '004
1032               Albania            '008
2139               Algeria            '012
3260        American Samoa            '016
3810               Andorra            '020
4498                Angola            '024
5544   Antigua and Barbuda            '028
6352             Argentina            '032
7409               Armenia            '051
8524             Australia            '036
9590               Austria            '040
10599           Azerbaijan            '031
11718              Bahamas            '044
12538              Bahrain            '048
13362           Bangladesh            '050

NOTE: M49 codes are numeric UN codes, NOT ISO alpha-3.


---
## 5. Quick attempt at joining FAO + World Bank

This validates whether the join is feasible at all.

In [33]:
import country_converter as coco

fao_area_codes = (
    fao_fs[['Area Code (M49)', 'Area']]
    .drop_duplicates()
    .copy()
)

# Remove apostrophe, then convert to numeric M49 code
fao_area_codes['m49_clean'] = (
    fao_area_codes['Area Code (M49)']
    .astype(str)
    .str.replace("'", "", regex=False)
    .str.strip()
)

fao_area_codes['m49_numeric'] = pd.to_numeric(
    fao_area_codes['m49_clean'],
    errors='coerce'
)

# Convert numeric UN/M49 code to ISO3
fao_area_codes['iso3'] = coco.convert(
    names=fao_area_codes['m49_numeric'].dropna().astype(int).tolist(),
    src='UNcode',
    to='ISO3',
    not_found=None
)

159 not found in UNcode
158 not found in UNcode
1 not found in UNcode
2 not found in UNcode
14 not found in UNcode
17 not found in UNcode
15 not found in UNcode
746 not found in UNcode
18 not found in UNcode
11 not found in UNcode
202 not found in UNcode
738 not found in UNcode
513 not found in UNcode
21 not found in UNcode
150 not found in UNcode
151 not found in UNcode
154 not found in UNcode
39 not found in UNcode
155 not found in UNcode
419 not found in UNcode
13 not found in UNcode
29 not found in UNcode
5 not found in UNcode
142 not found in UNcode
143 not found in UNcode
30 not found in UNcode
34 not found in UNcode
127 not found in UNcode
35 not found in UNcode
145 not found in UNcode
62 not found in UNcode
753 not found in UNcode
747 not found in UNcode
9 not found in UNcode
53 not found in UNcode
54 not found in UNcode
57 not found in UNcode
61 not found in UNcode
543 not found in UNcode
199 not found in UNcode
432 not found in UNcode
722 not found in UNcode
901 not found in 

In [28]:
fao_area_codes.head(10)

,Area Code (M49),Area,m49_clean,m49_numeric,iso3
0,'004,Afghanistan,004,4,AFG
1032,'008,Albania,008,8,ALB
2139,'012,Algeria,012,12,DZA
3260,'016,American Samoa,016,16,ASM
3810,'020,Andorra,020,20,AND
4498,'024,Angola,024,24,AGO
5544,'028,Antigua and Barbuda,028,28,ATG
6352,'032,Argentina,032,32,ARG
7409,'051,Armenia,051,51,ARM
8524,'036,Australia,036,36,AUS


In [31]:
# Verify that codes not found correspond to non-country areas (e.g. regions, aggregates)
fao_area_codes['valid_iso3'] = (
    fao_area_codes['iso3']
    .astype(str)
    .str.match(r'^[A-Z]{3}$')
)

fao_area_codes[fao_area_codes['valid_iso3'].ne(True)]

,Area Code (M49),Area,m49_clean,m49_numeric,iso3,valid_iso3
37160,'159,China,159,159,159,False
40498,'158,"China, Taiwan Province of",158,158,158,False
205540,'001,World,001,1,1,False
207257,'002,Africa,002,2,2,False
208974,'014,Eastern Africa,014,14,14,False
210691,'017,Middle Africa,017,17,17,False
212408,'015,Northern Africa,015,15,15,False
214134,'746,Northern Africa (excluding Sudan),746,746,746,False
215561,'018,Southern Africa,018,18,18,False
217230,'011,Western Africa,011,11,11,False


---
## 6. Open-Meteo: Climate feasibility check

Open-Meteo is point-based (lat/lon).
This cell tests one country's capital city as a representative point.

In [37]:
# Test: annual average temperature for Dominican Republic (Santo Domingo)
# Open-Meteo historical API, no auth required
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 18.4861,
    "longitude": -69.9312,
    "start_date": "2010-01-01",
    "end_date": "2020-12-31",
    "daily": "temperature_2m_mean,precipitation_sum",
    "timezone": "Africa/Addis_Ababa"
}

r = requests.get(OPEN_METEO_URL, params=params, timeout=60)
print(f"Status: {r.status_code}")

if r.status_code == 200:
    climate_data = r.json()
    climate_df = pd.DataFrame(climate_data['daily'])
    climate_df['date'] = pd.to_datetime(climate_df['time'])
    climate_df['year'] = climate_df['date'].dt.year

    annual = climate_df.groupby('year').agg(
        avg_temp=('temperature_2m_mean', 'mean'),
        total_precip=('precipitation_sum', 'sum')
    ).round(2)

    print("Annual climate data for Dominican Republic (Santo Domingo):")
    print(annual)
    print("\n(√) Open-Meteo works.")
else:
    print(f"(x) Open-Meteo failed: {r.text[:200]}")

Status: 200
Annual climate data for Dominican Republic (Santo Domingo):
      avg_temp  total_precip
year                        
2010     25.26        1123.9
2011     24.77        1111.1
2012     24.88        1038.2
2013     24.92         930.5
2014     25.18         672.3
2015     25.59         622.3
2016     25.25        1177.3
2017     25.43        1285.6
2018     25.14        1083.7
2019     25.69         982.5
2020     25.71        1252.1

(√) Open-Meteo works.


Since open Meteo is coordinate-level, the weather point is needed rather than the country name.
Strategy decision needed: use capital city coords per country, or multiple points?
 - Capital city is more convenient/easy, but it may not represent crop-growing regions.

---
## 7. Summary

### FAO Food Security (FS)
- [√] Download works? Size = 2.2 MB
- [√] Undernourishment data: 249 countries, years 2000 to 2024
- [√] Country code type: M49 numeric
- [√] Null/flag issues found: ___

### FAO Food Price Index (FPI)
- [√] Download works? Size = 2.3 MB  
- [√] Granularity: monthly / annual / both?
- [√] Countries covered: __ (note: global FPI vs country-level FPI)

### World Bank API
- [√] wbgapi works without auth
- [√] GDP null rate: 3% 
- [√] Poverty null rate: 66% --> It is survey-based and not available for every year and country.
- [√] Country code: ISO alpha-3

### Country Code Alignment
- [√] FAO M49 → ISO3 mapping: 100% exact matches with conversion using `country_converter`
- [-] Seed table needed: NO. Use `country_converter` instead.

### Open-Meteo (optional)
- [√] API works
- [√] Capital city strategy viable
- [-] Seed table needed: YES. With the coordinates of the capital cities of each country.

